# 📋 Notebook 1 — Data Cleaning
Load raw CSVs → inspect → clean → save to `data/processed/`

**Run this notebook FIRST before any other.**

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in CANDIDATE_ROOTS if (path / "src" / "helpers.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing src/helpers.py")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from helpers import RAW_DIR, PROCESSED_DIR, section

print("Libraries loaded")
print(f"Project root: {PROJECT_ROOT}")
print(f"Looking for raw files in: {RAW_DIR}")


Libraries loaded
Project root: c:\project\trading_sentiment_project
Looking for raw files in: C:\project\trading_sentiment_project\data\raw


## 1A — Load Fear & Greed Index

In [2]:
fg_path = os.path.join(RAW_DIR, 'fear_greed_index.csv')

if not os.path.exists(fg_path):
    raise FileNotFoundError(
        f'\n\n❌  File not found: {fg_path}'
        '\n    → Copy your fear_greed_index.csv into data/raw/ and rerun.'
    )

fg_raw = pd.read_csv(fg_path)
print(f'Shape: {fg_raw.shape}')
print(f'Columns: {fg_raw.columns.tolist()}')
fg_raw.head()

Shape: (2644, 4)
Columns: ['timestamp', 'value', 'classification', 'date']


,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [3]:
print('--- dtypes ---')
print(fg_raw.dtypes)
print('\n--- missing values ---')
print(fg_raw.isnull().sum())
print('\n--- duplicates ---')
print(fg_raw.duplicated().sum())

--- dtypes ---
timestamp          int64
value              int64
classification    object
date              object
dtype: object

--- missing values ---
timestamp         0
value             0
classification    0
date              0
dtype: int64

--- duplicates ---
0


## 1B — Clean Fear & Greed

In [4]:
fg = fg_raw.copy()

# ── Rename columns to standard names ──────────────────────────────
# Handle various possible column names from the dataset
rename_map = {}
for col in fg.columns:
    cl = col.strip().lower()
    if 'timestamp' in cl:       rename_map[col] = 'timestamp'
    elif cl == 'value':         rename_map[col] = 'value'
    elif 'class' in cl:        rename_map[col] = 'classification'
    elif 'date' in cl:         rename_map[col] = 'date'
fg.rename(columns=rename_map, inplace=True)

print('Renamed columns:', fg.columns.tolist())

Renamed columns: ['timestamp', 'value', 'classification', 'date']


In [5]:
# ── Parse date ────────────────────────────────────────────────────
if 'date' in fg.columns:
    fg['date'] = pd.to_datetime(fg['date'], errors='coerce')
elif 'timestamp' in fg.columns:
    # timestamp might be unix epoch
    try:
        fg['date'] = pd.to_datetime(fg['timestamp'], unit='s', errors='coerce')
    except:
        fg['date'] = pd.to_datetime(fg['timestamp'], errors='coerce')

# ── Drop rows with no date or no value ───────────────────────────
before = len(fg)
fg.dropna(subset=['date', 'value'], inplace=True)
print(f'Dropped {before - len(fg)} rows with null date/value')

# ── Remove duplicates (keep last) ────────────────────────────────
before = len(fg)
fg.drop_duplicates(subset=['date'], keep='last', inplace=True)
print(f'Dropped {before - len(fg)} duplicate dates')

# ── Ensure value is numeric and in 0-100 ─────────────────────────
fg['value'] = pd.to_numeric(fg['value'], errors='coerce')
out_of_range = fg[(fg['value'] < 0) | (fg['value'] > 100)]
print(f'Out-of-range values (0-100): {len(out_of_range)}')
fg = fg[(fg['value'] >= 0) & (fg['value'] <= 100)]

# ── Standardise classification labels ────────────────────────────
valid_zones = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
fg['classification'] = fg['classification'].astype(str).str.strip().str.title()

# Fix common variations
fix = {
    'Extreme Fear': 'Extreme Fear',
    'Extremefear':  'Extreme Fear',
    'Extreme Greed':'Extreme Greed',
    'Extremegreed': 'Extreme Greed',
}
fg['classification'] = fg['classification'].replace(fix)

# Re-classify by numeric value if classification is missing/bad
def classify_by_value(v):
    if v <= 24:  return 'Extreme Fear'
    elif v <= 44: return 'Fear'
    elif v <= 55: return 'Neutral'
    elif v <= 74: return 'Greed'
    else:         return 'Extreme Greed'

bad_mask = ~fg['classification'].isin(valid_zones)
fg.loc[bad_mask, 'classification'] = fg.loc[bad_mask, 'value'].apply(classify_by_value)
print(f'Re-classified {bad_mask.sum()} rows with bad labels')

# ── Add helper columns ────────────────────────────────────────────
fg['year']  = fg['date'].dt.year
fg['month'] = fg['date'].dt.month
fg['month_name'] = fg['date'].dt.strftime('%b %Y')

# Zone as ordered category
fg['zone'] = pd.Categorical(
    fg['classification'],
    categories=valid_zones,
    ordered=True
)

fg.sort_values('date', inplace=True)
fg.reset_index(drop=True, inplace=True)

print(f'\nCleaned Fear & Greed shape: {fg.shape}')
print(f'Date range: {fg["date"].min().date()} → {fg["date"].max().date()}')
print('\nZone distribution:')
print(fg['classification'].value_counts())
fg.head()

Dropped 0 rows with null date/value
Dropped 0 duplicate dates
Out-of-range values (0-100): 0
Re-classified 0 rows with bad labels

Cleaned Fear & Greed shape: (2644, 8)
Date range: 2018-02-01 → 2025-05-02

Zone distribution:
classification
Fear             781
Greed            633
Extreme Fear     508
Neutral          396
Extreme Greed    326
Name: count, dtype: int64


,timestamp,value,classification,date,year,month,month_name,zone
0,1517463000,30,Fear,2018-02-01,2018,2,Feb 2018,Fear
1,1517549400,15,Extreme Fear,2018-02-02,2018,2,Feb 2018,Extreme Fear
2,1517635800,40,Fear,2018-02-03,2018,2,Feb 2018,Fear
3,1517722200,24,Extreme Fear,2018-02-04,2018,2,Feb 2018,Extreme Fear
4,1517808600,11,Extreme Fear,2018-02-05,2018,2,Feb 2018,Extreme Fear


## 1C — Load Trades Dataset

In [6]:
tr_path = os.path.join(RAW_DIR, 'historical_trades.csv')

if not os.path.exists(tr_path):
    raise FileNotFoundError(
        f'\n\n❌  File not found: {tr_path}'
        '\n    → Copy your historical_trades.csv into data/raw/ and rerun.'
    )

tr_raw = pd.read_csv(tr_path, low_memory=False)
print(f'Shape: {tr_raw.shape}')
print(f'Columns: {tr_raw.columns.tolist()}')
tr_raw.head()

Shape: (211224, 16)
Columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [7]:
print('--- dtypes ---')
print(tr_raw.dtypes)
print('\n--- missing values (top 15) ---')
print(tr_raw.isnull().sum().sort_values(ascending=False).head(15))
print('\n--- duplicates ---')
print(tr_raw.duplicated().sum())
print('\n--- numeric column stats ---')
tr_raw.describe()

--- dtypes ---
Account              object
Coin                 object
Execution Price     float64
Size Tokens         float64
Size USD            float64
Side                 object
Timestamp IST        object
Start Position      float64
Direction            object
Closed PnL          float64
Transaction Hash     object
Order ID              int64
Crossed                bool
Fee                 float64
Trade ID            float64
Timestamp           float64
dtype: object

--- missing values (top 15) ---
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
dtype: int64

--- duplicates ---
0

--- numeric column stats ---


,Execution Price,Size Tokens,Size USD,Start Position,Closed PnL,Order ID,Fee,Trade ID,Timestamp
count,211224.000000,2.112240e+05,2.112240e+05,2.112240e+05,211224.000000,2.112240e+05,211224.000000,2.112240e+05,2.112240e+05
mean,11414.723350,4.623365e+03,5.639451e+03,-2.994625e+04,48.749001,6.965388e+10,1.163967,5.628549e+14,1.737744e+12
std,29447.654868,1.042729e+05,3.657514e+04,6.738074e+05,919.164828,1.835753e+10,6.758854,3.257565e+14,8.689920e+09
min,0.000005,8.740000e-07,0.000000e+00,-1.433463e+07,-117990.104100,1.732711e+08,-1.175712,0.000000e+00,1.680000e+12
25%,4.854700,2.940000e+00,1.937900e+02,-3.762311e+02,0.000000,5.983853e+10,0.016121,2.810000e+14,1.740000e+12
50%,18.280000,3.200000e+01,5.970450e+02,8.472793e+01,0.000000,7.442939e+10,0.089578,5.620000e+14,1.740000e+12
75%,101.580000,1.879025e+02,2.058960e+03,9.337278e+03,5.792797,8.335543e+10,0.393811,8.460000e+14,1.740000e+12
max,109004.000000,1.582244e+07,3.921431e+06,3.050948e+07,135329.090100,9.014923e+10,837.471593,1.130000e+15,1.750000e+12


## 1D — Clean Trades Dataset

In [8]:
tr = tr_raw.copy()

tr.columns = (
    tr.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace(r"[^a-z0-9_]", "", regex=True)
)

rename_map = {
    "account": "account",
    "coin": "symbol",
    "execution_price": "exec_price",
    "size_tokens": "size_tokens",
    "size_usd": "size_usd",
    "side": "side",
    "timestamp_ist": "time",
    "start_position": "start_position",
    "direction": "direction",
    "closed_pnl": "closedPnL",
    "transaction_hash": "transaction_hash",
    "order_id": "order_id",
    "crossed": "crossed",
    "fee": "fee",
    "trade_id": "trade_id",
    "timestamp": "batch_timestamp",
}
tr.rename(columns=rename_map, inplace=True)

expected_cols = [
    "account",
    "symbol",
    "exec_price",
    "size_tokens",
    "size_usd",
    "side",
    "time",
    "start_position",
    "direction",
    "closedPnL",
    "transaction_hash",
    "order_id",
    "crossed",
    "fee",
    "trade_id",
    "batch_timestamp",
]
missing_cols = [col for col in expected_cols if col not in tr.columns]
if missing_cols:
    raise KeyError(f"Missing expected trade columns: {missing_cols}")

print("Final columns:", tr.columns.tolist())
print("\nSample trade rows:")
print(tr[["symbol", "side", "direction", "time"]].head(5).to_string(index=False))


Final columns: ['account', 'symbol', 'exec_price', 'size_tokens', 'size_usd', 'side', 'time', 'start_position', 'direction', 'closedPnL', 'transaction_hash', 'order_id', 'crossed', 'fee', 'trade_id', 'batch_timestamp']

Sample trade rows:
symbol side direction             time
  @107  BUY       Buy 02-12-2024 22:50
  @107  BUY       Buy 02-12-2024 22:50
  @107  BUY       Buy 02-12-2024 22:50
  @107  BUY       Buy 02-12-2024 22:50
  @107  BUY       Buy 02-12-2024 22:50


In [9]:
print("Raw time samples:")
print(tr["time"].head(10).tolist())

raw_time = tr["time"].astype(str).str.strip()
tr["time"] = pd.to_datetime(raw_time, format="%d-%m-%Y %H:%M", errors="coerce")

fallback_mask = tr["time"].isna()
if fallback_mask.any():
    tr.loc[fallback_mask, "time"] = pd.to_datetime(
        raw_time[fallback_mask],
        dayfirst=True,
        errors="coerce",
    )

tr["date"] = tr["time"].dt.normalize()

print("\nAfter parsing:")
print(f"Unique dates  : {tr['date'].nunique()}")
print(f"Date range    : {tr['date'].min()} -> {tr['date'].max()}")
print(f"Null times    : {tr['time'].isna().sum()}")
print("\nSample parsed dates:")
print(tr[["time", "date"]].head(10).to_string(index=False))


Raw time samples:
['02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50']

After parsing:
Unique dates  : 480
Date range    : 2023-05-01 00:00:00 -> 2025-05-01 00:00:00
Null times    : 0

Sample parsed dates:
               time       date
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02
2024-12-02 22:50:00 2024-12-02


In [10]:
before = len(tr)
tr.dropna(how="all", inplace=True)
print(f"Dropped {before - len(tr)} fully empty rows")

before = len(tr)
tr.drop_duplicates(inplace=True)
print(f"Dropped {before - len(tr)} duplicate rows")

numeric_cols = [
    "exec_price",
    "size_tokens",
    "size_usd",
    "start_position",
    "closedPnL",
    "fee",
    "order_id",
    "trade_id",
    "batch_timestamp",
]
for col in numeric_cols:
    tr[col] = pd.to_numeric(tr[col], errors="coerce")

tr["size_tokens"] = tr["size_tokens"].abs()
tr["size_usd"] = tr["size_usd"].abs()

before = len(tr)
tr = tr[tr["exec_price"] > 0].copy()
print(f"Dropped {before - len(tr)} rows with non-positive execution prices")

side_map = {
    "BUY": "BUY",
    "B": "BUY",
    "LONG": "BUY",
    "SELL": "SELL",
    "S": "SELL",
    "SHORT": "SELL",
}
tr["side"] = tr["side"].astype(str).str.strip().str.upper().map(side_map)
tr["direction"] = tr["direction"].astype(str).str.strip().str.upper().map(side_map)
side_mismatches = ((tr["side"].notna()) & (tr["direction"].notna()) & (tr["side"] != tr["direction"])).sum()
tr["side"] = tr["side"].fillna(tr["direction"])
print(f"Side/direction mismatches: {side_mismatches}")
print("Side values:", sorted(tr["side"].dropna().unique().tolist()))

for col in ["order_id", "trade_id", "batch_timestamp"]:
    tr[col] = tr[col].round().astype("Int64")

tr["is_win"] = tr["closedPnL"] > 0

tr.sort_values(["date", "time", "order_id", "trade_id"], inplace=True)
tr.reset_index(drop=True, inplace=True)

print(f"\nCleaned trades shape: {tr.shape}")
print(tr.head().to_string())


Dropped 0 fully empty rows
Dropped 0 duplicate rows
Dropped 0 rows with non-positive execution prices
Side/direction mismatches: 0
Side values: ['BUY', 'SELL']

Cleaned trades shape: (211224, 18)
                                      account symbol  exec_price  size_tokens  size_usd side                time  start_position direction  closedPnL                                                    transaction_hash    order_id  crossed       fee         trade_id  batch_timestamp       date  is_win
0  0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891    ETH      1897.9       0.0967    183.53  BUY 2023-05-01 01:06:00          0.0000       NaN        0.0  0x875d3e1af52b5b758e4f04015b774e0111006a118601535a3fa23953792605a2   173271100     True  0.000000                0    1680000000000 2023-05-01   False
1  0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891    ETH      1897.9       0.0824    156.39  BUY 2023-05-01 01:06:00          0.0967       NaN        0.0  0x875d3e1af52b5b758e4f04015b774e0111006a118601535

## 1E — Merge Datasets on Date

In [11]:
fg_for_merge = fg[["date", "value", "classification", "zone"]].copy()
fg_for_merge.rename(columns={"value": "fgi_score"}, inplace=True)

merged = tr.merge(fg_for_merge, on="date", how="left", validate="many_to_one")
merged["has_sentiment"] = merged["fgi_score"].notna()

unmatched_rows = (~merged["has_sentiment"]).sum()
unmatched_dates = (
    merged.loc[~merged["has_sentiment"], "date"]
          .dropna()
          .drop_duplicates()
          .sort_values()
)

print(f"Trades matched to FGI score: {merged['has_sentiment'].sum():,} / {len(merged):,}")
print(f"Unmatched rows retained    : {unmatched_rows:,}")
if len(unmatched_dates):
    print("Dates missing an FGI observation:")
    print(unmatched_dates.dt.strftime("%Y-%m-%d").to_list())

analysis_ready = merged[merged["has_sentiment"]].copy()
print(f"Analysis-ready matched rows: {len(analysis_ready):,}")
print(f"Final merged shape         : {merged.shape}")
print(merged.head().to_string())


Trades matched to FGI score: 211,218 / 211,224
Unmatched rows retained    : 6
Dates missing an FGI observation:
['2024-10-26']
Analysis-ready matched rows: 211,218
Final merged shape         : (211224, 22)
                                      account symbol  exec_price  size_tokens  size_usd side                time  start_position direction  closedPnL                                                    transaction_hash    order_id  crossed       fee         trade_id  batch_timestamp       date  is_win  fgi_score classification           zone  has_sentiment
0  0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891    ETH      1897.9       0.0967    183.53  BUY 2023-05-01 01:06:00          0.0000       NaN        0.0  0x875d3e1af52b5b758e4f04015b774e0111006a118601535a3fa23953792605a2   173271100     True  0.000000                0    1680000000000 2023-05-01   False       63.0          Greed          Greed           True
1  0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891    ETH      1897.9       0.0824 

## 1F — Save Cleaned Files

In [12]:
fg.to_csv(PROCESSED_DIR / "fear_greed_clean.csv", index=False)
tr.to_csv(PROCESSED_DIR / "trades_clean.csv", index=False)
merged.to_csv(PROCESSED_DIR / "merged.csv", index=False)

print("Saved to data/processed/")
print(f"  fear_greed_clean.csv -> {fg.shape}")
print(f"  trades_clean.csv     -> {tr.shape}")
print(f"  merged.csv           -> {merged.shape}")


Saved to data/processed/
  fear_greed_clean.csv -> (2644, 8)
  trades_clean.csv     -> (211224, 18)
  merged.csv           -> (211224, 22)


In [13]:
print("=== CLEANING SUMMARY ===")
print(f"Fear & Greed Index records : {len(fg):,}")
print(f"  Date range               : {fg['date'].min().date()} -> {fg['date'].max().date()}")
print(f"  Missing values left      : {fg.isnull().sum().sum()}")
print()
print(f"Trades records             : {len(tr):,}")
print(f"  Date range               : {tr['date'].min().date()} -> {tr['date'].max().date()}")
print(f"  Missing parsed times     : {tr['time'].isna().sum():,}")
print()
print(f"Merged records             : {len(merged):,}")
print(f"  Matched sentiment rows   : {merged['has_sentiment'].sum():,}")
print(f"  Unmatched rows retained  : {(~merged['has_sentiment']).sum():,}")
print()
print("Run Notebook 02 next.")


=== CLEANING SUMMARY ===
Fear & Greed Index records : 2,644
  Date range               : 2018-02-01 -> 2025-05-02
  Missing values left      : 0

Trades records             : 211,224
  Date range               : 2023-05-01 -> 2025-05-01
  Missing parsed times     : 0

Merged records             : 211,224
  Matched sentiment rows   : 211,218
  Unmatched rows retained  : 6

Run Notebook 02 next.
